# Calibration ablation + predicted groups, part 2: DINOv2 and ViT-B/16

The label-conditional set fix invalidated every Mondrian set size, so
`revision_calibration_4bb.ipynb` is rebuilding all eight cells of both studies in one runtime.
That is ~7 h serial. This notebook takes the two backbones that runtime has not reached yet and
runs them in a **second** runtime, beside it.

**The one rule that makes this safe.** Both streaming runs rewrite their whole CSV after every
finished cell, so two runtimes pointed at one file would overwrite each other. This notebook
writes `*_part2.csv` and never touches the canonical files. `tools/merge_records.py` merges the
two halves afterwards, deduplicating by (backbone, dataset, method, seed, ...) so an accidentally
repeated cell cannot double-count.

It also never archives anything: the other runtime already did that, and re-archiving would move
a file it is in the middle of writing.

## 0. Parameters

In [ ]:
REPO_URL      = "https://github.com/octadion/vgscp"
REPO_BRANCH   = "main"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
WATERBIRDS_URL= "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE = "kaggle"       # the Drive-cached zip is used first; no credential needed
CELEBA_DRIVE  = ""

# The half this runtime owns. The other runtime works ResNet-50 and CLIP.
BACKBONES = ("dinov2_vitb14", "vit_b16_in1k")
DATASETS  = ("waterbirds", "celeba")

# Anything the other runtime is inside right now, so this one does not repeat hours of work.
# Runtime 1 walks waterbirds first, then celeba, in BACKBONES order.
SKIP_KEYS = ()                 # e.g. (("clip_vitb32", "celeba"),)

METHODS   = ("erm", "dfr", "afr", "groupdro_ll", "balanced_subsample")
SCORES    = ("APS", "RAPS", "THR")
RHO_SWEEP = (0.95, 0.9, 0.8, 0.7, 0.6, 0.5)
CAL_SEEDS = (0, 1, 2)          # must match the other runtime, or the tables mix protocols
N_SPLITS  = 10
ALPHA     = 0.1

CELEBA_RESNET_MAX_TRAIN = 30000   # part of the cache key even though no ResNet cell runs here

CAL_CSV = "results/study/calibration_ablation_4bb_part2.csv"
PG_CSV  = "results/study/predicted_group_mondrian_4bb_part2.csv"

## 1. Drive, repo, caches

In [ ]:
import os, sys, time, subprocess

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(f"[shell FAILED rc={r.returncode}] {cmd}")
        if r.stderr.strip(): print(r.stderr[-2000:])
        if check: raise RuntimeError(f"command failed: {cmd}")
    return r.returncode == 0

from google.colab import drive
drive.mount("/content/drive"); os.makedirs(DRIVE_CACHE, exist_ok=True)

REPO_DIR = "/content/vgscp"
sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

os.makedirs("results", exist_ok=True)
for c in ("cache_clip", "cache_resnet", "cache_frozen", "study"):
    tgt = f"{DRIVE_CACHE}/{c}"; os.makedirs(tgt, exist_ok=True)
    sh(f"rm -rf results/{c}"); sh(f"ln -s {tgt} results/{c}")
    probe = f"results/{c}/.link_probe_part2"
    with open(probe, "w") as fh: fh.write("ok")
    assert os.path.exists(f"{tgt}/.link_probe_part2"), f"results/{c} does not resolve to Drive"
    os.remove(probe)

ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
print(f"\nrepo: {os.getcwd()}\nvCPUs: {os.cpu_count()}  RAM: {ram:.1f} GB")
print("DINOv2 (d=768) and ViT-B/16 (d=768) are the two cheapest cells; the other runtime has the")
print("2048-d ResNet, so both fit comfortably.")

## 2. Datasets (paths only -- they are hashed into the cache key)

In [ ]:
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
assert CELEBA_ROOT and os.path.isdir(CELEBA_ROOT), "CelebA unavailable"
os.environ["CELEBA_ROOT"] = CELEBA_ROOT
print("datasets ready |", os.environ["WATERBIRDS_ROOT"], "|", CELEBA_ROOT)

## 3. Gate: this clone must carry the label-conditional set fix

Same guard as the other notebook. Nothing is archived here.

In [ ]:
rc = subprocess.run([sys.executable, "-m", "study_robust_train.validate_mondrian_sets"],
                    capture_output=True, text=True)
print(rc.stdout.strip() or rc.stderr[-800:])
assert rc.returncode == 0, ("this clone predates the label-conditional set fix -- push it to "
                            f"{REPO_BRANCH}, then re-run section 1")

## 4. What is left to do

Reads the canonical CSVs the other runtime is writing and this notebook's own part-2 files, and
prints which (backbone, dataset) cells still have no records anywhere. The canonical file is being
rewritten while you read it, so a torn read is possible and is reported rather than trusted: if a
cell shows up as missing here but the other runtime already did it, the merge will dedupe it.

If the other runtime is heading for the same cells, interrupt it once its current cell prints
`[cell done]` -- every finished cell is already on Drive, so nothing is lost -- and let this one
take them.

In [ ]:
from study_robust_train.calibration_ablation import records_from_csv as cal_records_from_csv
from study_robust_train.predicted_group_mondrian import records_from_csv as pg_records_from_csv

def cells_in(path, reader):
    if not os.path.exists(path):
        return set(), "absent"
    try:
        return {(r["backbone"], r["dataset"]) for r in reader(path)}, "read"
    except Exception as e:                      # torn read while the other runtime writes
        return set(), f"unreadable ({type(e).__name__})"

want = [(bb, ds) for ds in DATASETS for bb in BACKBONES if (bb, ds) not in SKIP_KEYS]

plan = {}
for label, canon, part2, reader in (
        ("ablation", "results/study/calibration_ablation_4bb.csv", CAL_CSV, cal_records_from_csv),
        ("predicted-group", "results/study/predicted_group_mondrian_4bb.csv", PG_CSV,
         pg_records_from_csv)):
    a, sa = cells_in(canon, reader)
    b, sb = cells_in(part2, reader)
    todo = [k for k in want if k not in a and k not in b]
    plan[label] = todo
    print(f"{label}:")
    print(f"   canonical [{sa}] {sorted(a) if a else '--'}")
    print(f"   part2     [{sb}] {sorted(b) if b else '--'}")
    print(f"   THIS RUNTIME WILL RUN: {todo if todo else 'nothing left'}")
    print()

## 5. Build one cell at a time (cache hit expected: seconds, not minutes)

In [ ]:
import gc
from study_robust_train.datasets import build_griddata

def cfg_for(dataset):
    base = {"clip":   {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cpu",
                       "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cpu", "epochs": 10, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"},
            "frozen": {"device": "cpu", "cache_dir": "results/cache_frozen",
                       "batch_size": 128, "num_workers": 4}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
        base["resnet"]["max_train"] = CELEBA_RESNET_MAX_TRAIN
    return base

SLOW_SECONDS = 300      # Drive is serving two runtimes now, so allow more than the other notebook

def build_cell(bb, ds):
    t = time.time()
    gd = build_griddata(ds, bb, cfg_for(ds), seed=0)
    el = time.time() - t
    print(f"[loaded] {bb:14s}/{ds:10s} d={gd.train[0].shape[1]:5d} "
          f"train={gd.train[0].shape[0]:6d} eval={gd.eval_domain[0].shape[0]:6d} ({el:.0f}s)",
          flush=True)
    assert el < SLOW_SECONDS, (f"{bb}/{ds} took {el:.0f}s -- COMPUTED, not loaded. That cache is "
                               f"missing; do not let a CPU runtime extract features.")
    return gd

for ds in DATASETS:                       # probe both datasets on the cheaper backbone
    gd = build_cell(BACKBONES[0], ds); del gd; gc.collect()
print("\ncaches verified")

## 6. Ablation, this half only (resumable: re-run the cell after a disconnect)

In [ ]:
from study_robust_train.calibration_ablation import run_ablation_streaming

if not plan["ablation"]:
    print("the other runtime already has every ablation cell -- nothing to do here")
else:
    t = time.time()
    cal = run_ablation_streaming(plan["ablation"], build_cell, methods=METHODS, scores=SCORES,
                                 rho_sweep=RHO_SWEEP, seeds=CAL_SEEDS, n_splits=N_SPLITS,
                                 alpha=ALPHA, cell_csv=CAL_CSV)
    print(f"\nelapsed {(time.time() - t) / 60:.0f} min | records {len(cal['records']):,}")
    by = {}
    for r in cal["records"]:
        by[r["gate_status"]] = by.get(r["gate_status"], 0) + 1
    print("gate_status:", by)
    done = sorted({(r["backbone"], r["dataset"]) for r in cal["records"]})
    print(f"cells in {os.path.basename(CAL_CSV)}: {done}")
    if [k for k in plan["ablation"] if k not in done]:
        print("Re-run THIS cell to continue.")

## 7. Predicted-group Mondrian, this half only

In [ ]:
from study_robust_train.predicted_group_mondrian import run_predicted_group_streaming

if not plan["predicted-group"]:
    print("the other runtime already has every predicted-group cell -- nothing to do here")
else:
    t = time.time()
    pg = run_predicted_group_streaming(plan["predicted-group"], build_cell, methods=METHODS,
                                       scores=SCORES, seeds=CAL_SEEDS, n_splits=N_SPLITS,
                                       alpha=ALPHA, cell_csv=PG_CSV)
    print(f"\nelapsed {(time.time() - t) / 60:.0f} min | records {len(pg['records']):,}")
    done = sorted({(r["backbone"], r["dataset"]) for r in pg["records"]})
    print(f"cells in {os.path.basename(PG_CSV)}: {done}")
    if [k for k in plan["predicted-group"] if k not in done]:
        print("Re-run THIS cell to continue.")

## 8. Hand back

Download both `*_part2.csv` files (they are on Drive under `vgscp_cache/study/`) together with the
canonical two the other runtime writes. Locally:

```
python tools/merge_records.py ablation  results/calibration_ablation_4bb.csv
python tools/merge_records.py predgroup results/predicted_group_mondrian_4bb.csv
```

The merge refuses to write unless all eight cells are present in each study, so a half-finished
run cannot quietly become the paper's numbers.

In [ ]:
sh(f"ls -la {DRIVE_CACHE}/study/", check=False)